# データ保存先

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# Arase

In [ ]:
import pyspedas as psp
import pytplot as pt
import ergpyspedas.erg as ergpy

pt.del_data('*')

time_range = ['20220901/21:00:00', '20220902/00:00:00']

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/2230-2330'

ergpy.pwe_efd(trange=time_range, level='l2', datatype='64', coord='dsi', get_support_data=True, no_update=True)
ergpy.mgf(trange=time_range, level='l2', datatype='64hz', coord='dsi', get_support_data=True, no_update=True)

print("--- Loaded tplot variables ---")
print(psp.tplot_names())

In [ ]:
import xarray as xr
import numpy as np

time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]

B64_data_dsi    = psp.get_data('erg_mgf_l2_mag_64hz_dsi', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B64_data_dsi_quality_flag   = psp.get_data('erg_mgf_l2_quality_64hz', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

qf_B, B64 = xr.align(B64_data_dsi_quality_flag[:, 3], B64_data_dsi, join='inner')

B64_data_dsi_qf = xr.where(qf_B <= 21, B64, np.nan)

ds_B64_dsi  = xr.Dataset({
    'B64_dsi_x':    B64_data_dsi_qf[:, 0],
    'B64_dsi_y':    B64_data_dsi_qf[:, 1],
    'B64_dsi_z':    B64_data_dsi_qf[:, 2]
})

ds_B64_dsi  = ds_B64_dsi.dropna(dim='time', how='all')

In [ ]:
def split_by_gap(ds, time_dim='time', gap_thr=np.timedelta64(30, 's'),
                 prefix='ds_8_gsm_seg'):
    t = ds[time_dim].values
    if t.size == 0:
        return []

    # 先頭はギャップなし。以降で gap_thr を超えたら新セグ開始。
    gaps = np.r_[False, (t[1:] - t[:-1]) > gap_thr]      # shape (Nt,)
    seg_id = np.cumsum(gaps)                              # 0,0,0,1,1,2,...

    ds_tagged = ds.assign_coords(_seg=(time_dim, seg_id))
    segs = [g.drop_vars('_seg') for _, g in ds_tagged.groupby('_seg')]

    return segs

In [ ]:
ds_B64_dsi_segs = split_by_gap(ds_B64_dsi, gap_thr=np.timedelta64(63, 'ms'))
print(len(ds_B64_dsi_segs))

In [ ]:
ds_B64_dsi_seg0 = ds_B64_dsi_segs[0]
print(ds_B64_dsi_seg0)

In [ ]:
da_mgf_spin_phase_deg           = psp.get_data('erg_mgf_l2_spin_phase_64hz', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
da_mgf_spin_phase_deg_interp    = da_mgf_spin_phase_deg.interp(time=ds_B64_dsi.time)
da_mgf_spin_phase_rad_interp    = np.deg2rad(da_mgf_spin_phase_deg_interp)

In [ ]:
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.erg_mgf_spintone_rm as emsr
importlib.reload(emsr)
os.chdir('./KAW_observation')
print(os.getcwd())

B_clean_ndarray, B_spt_ndarray, params  = emsr.remove_spintone_3comp(
    time=ds_B64_dsi_seg0.time.values,
    Bx=ds_B64_dsi_seg0['B64_dsi_x'].values,
    By=ds_B64_dsi_seg0['B64_dsi_y'].values,
    Bz=ds_B64_dsi_seg0['B64_dsi_z'].values,
    phase_rad=da_mgf_spin_phase_rad_interp.values,
    min_points=64.*3./2.
)

ds_B64_dsi_seg0_spt      = xr.Dataset({
    'B64_dsi_x_spt':    ('time', B_spt_ndarray[:, 0]),
    'B64_dsi_y_spt':    ('time', B_spt_ndarray[:, 1]),
    'B64_dsi_z_spt':    ('time', B_spt_ndarray[:, 2])
    }, coords={'time': ds_B64_dsi_seg0.time.values})

ds_B64_dsi_seg0_clean    = xr.Dataset({
    'B64_dsi_x_clean':    ('time', B_clean_ndarray[:, 0]),
    'B64_dsi_y_clean':    ('time', B_clean_ndarray[:, 1]),
    'B64_dsi_z_clean':    ('time', B_clean_ndarray[:, 2])
    }, coords={'time': ds_B64_dsi_seg0.time.values})

print(ds_B64_dsi_seg0_spt)
print(ds_B64_dsi_seg0_clean)

In [ ]:
E64_data_dsi_x  = psp.get_data('erg_pwe_efd_l2_E64Hz_dsi_Ex_waveform', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_dsi_y  = psp.get_data('erg_pwe_efd_l2_E64Hz_dsi_Ey_waveform', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_dsi_quality_flag = psp.get_data('erg_pwe_efd_l2_E64Hz_dsi_quality_flag', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

# QF = 0 の時間を抽出
bad_times = E64_data_dsi_quality_flag.time.where(E64_data_dsi_quality_flag != 0, drop=True)

E64_data_dsi_x_qf = E64_data_dsi_x.where(E64_data_dsi_quality_flag.interp(time=E64_data_dsi_x.time, method='nearest', kwargs={'fill_value': 'extrapolate'}) == 0, np.nan)
E64_data_dsi_y_qf = E64_data_dsi_y.where(E64_data_dsi_quality_flag.interp(time=E64_data_dsi_y.time, method='nearest', kwargs={'fill_value': 'extrapolate'}) == 0, np.nan)

ds_E64_dsi_xy   = xr.Dataset({
    'E64_dsi_x': E64_data_dsi_x_qf,
    'E64_dsi_y': E64_data_dsi_y_qf
})
ds_E64_dsi_xy   = ds_E64_dsi_xy.dropna(dim='time', how='all')

In [ ]:
ds_E64_dsi_xy_segs = split_by_gap(ds_E64_dsi_xy, gap_thr=np.timedelta64(63, 'ms'))
print(ds_E64_dsi_xy_segs)

In [ ]:
def make_ds_EB_func(ds_E_xy, E_vars, ds_B_xyz, B_vars, output_vars):
    time_base   = ds_E_xy.time
    ds_B_xyz_interp  = ds_B_xyz.interp(time=time_base, method='linear')

    da_Ex   = ds_E_xy[E_vars[0]]
    da_Ey   = ds_E_xy[E_vars[1]]
    da_Bx   = ds_B_xyz_interp[B_vars[0]]
    da_By   = ds_B_xyz_interp[B_vars[1]]
    da_Bz   = ds_B_xyz_interp[B_vars[2]]

    da_Ez   = xr.where(np.abs(da_Bz) > 1E-2, -(da_Ex * da_Bx + da_Ey * da_By) / da_Bz, np.nan)

    ds_EB   = xr.Dataset({
        output_vars[0]: da_Ex,
        output_vars[1]: da_Ey,
        output_vars[2]: da_Ez,
        output_vars[3]: da_Bx,
        output_vars[4]: da_By,
        output_vars[5]: da_Bz,
    })

    ds_EB   = ds_EB.dropna(dim='time', how='any')

    return ds_EB

In [ ]:
E64_xy_vars     = ['E64_dsi_x', 'E64_dsi_y']
B64_xyz_vars    = ['B64_dsi_x_clean', 'B64_dsi_y_clean', 'B64_dsi_z_clean']
EB64_vars       = ['E64_dsi_x', 'E64_dsi_y', 'E64_dsi_z', 'B64_dsi_x', 'B64_dsi_y', 'B64_dsi_z']

In [ ]:
ds_EB64_dsi_segs    = []

for count in range(len(ds_E64_dsi_xy_segs)):
    ds_     = make_ds_EB_func(ds_E64_dsi_xy_segs[count], E64_xy_vars, ds_B64_dsi_seg0_clean, B64_xyz_vars, EB64_vars)
    ds_EB64_dsi_segs.append(ds_)

print(ds_EB64_dsi_segs)

In [ ]:
ds_EB64_dsi_segs = [ds_EB64_dsi_segs[0], ds_EB64_dsi_segs[2], ds_EB64_dsi_segs[3], ds_EB64_dsi_segs[4], ds_EB64_dsi_segs[6]]

In [ ]:
print(ds_EB64_dsi_segs)

In [ ]:
path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/2230-2330'
os.makedirs(path_base_save_plot, exist_ok=True)

In [ ]:
ergpy.orb(trange=time_range, level='l2', datatype='def')
da_Arase_pos_gsm    = psp.get_data('erg_orb_l2_pos_gsm', xarray=True)
da_Arase_pos_gsm    = da_Arase_pos_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

da_Arase_pos_unit_gsm   = da_Arase_pos_gsm / np.sqrt((da_Arase_pos_gsm * da_Arase_pos_gsm).sum(dim='v_dim'))

psp.store_data('Arase_pos_unit_gsm', data={'x': da_Arase_pos_unit_gsm.time, 'y': da_Arase_pos_unit_gsm.data})
psp.cotrans(name_in='Arase_pos_unit_gsm', name_out='Arase_pos_unit_j2000', coord_in='gsm', coord_out='j2000')
psp.projects.erg.erg_cotrans(in_name='Arase_pos_unit_j2000', out_name='Arase_pos_unit_dsi', in_coord='j2000', out_coord='dsi')

da_Arase_pos_unit_dsi   = psp.get_data('Arase_pos_unit_dsi', xarray=True)

In [ ]:
background_time_sec = 100 #[sec]

In [ ]:
time_width_B64          = (ds_B64_dsi_seg0_clean.time[10] - ds_B64_dsi_seg0_clean.time[9]) / np.timedelta64(1, 's')
ds_B_background         = ds_B64_dsi_seg0_clean.rolling(time=int(background_time_sec / time_width_B64), center=True).mean('time')

da_B_background         = ds_B_background.to_dataarray(dim='v_dim').T.dropna(dim='time', how='any')

print(da_B_background)

da_B_background_unit    = da_B_background / np.sqrt((da_B_background * da_B_background).sum(dim='v_dim'))
da_B_background_unit    = da_B_background_unit.dropna(how='all', dim='time')

print(da_B_background_unit)
print(np.nanmin(np.sqrt((da_B_background_unit * da_B_background_unit).sum(dim='v_dim'))))
print(np.nanmax(np.sqrt((da_B_background_unit * da_B_background_unit).sum(dim='v_dim'))))

In [ ]:
da_Arase_pos_unit_dsi_interp    = da_Arase_pos_unit_dsi.interp(time=da_B_background_unit.time)

da_u_   = da_Arase_pos_unit_dsi_interp - (da_Arase_pos_unit_dsi_interp * da_B_background_unit).sum(dim='v_dim') * da_B_background_unit

da_e_z_FAC_inDSI    = da_B_background_unit.drop_attrs()
da_e_x_FAC_inDSI    = (da_u_ / np.sqrt((da_u_ * da_u_).sum(dim='v_dim'))).drop_attrs()
da_e_y_FAC_inDSI    = (xr.apply_ufunc(np.cross, da_e_z_FAC_inDSI, da_e_x_FAC_inDSI, input_core_dims=[['v_dim'], ['v_dim']], output_core_dims=[['v_dim']], vectorize=True)).drop_attrs()

print(da_e_x_FAC_inDSI)
print('')
print(da_e_y_FAC_inDSI)
print('')
print(da_e_z_FAC_inDSI)

In [ ]:
R_FAC_to_DSI = xr.concat(
    [da_e_x_FAC_inDSI, da_e_y_FAC_inDSI, da_e_z_FAC_inDSI],
    dim='axis'
)
R_FAC_to_DSI = R_FAC_to_DSI.assign_coords(axis=['x_FAC', 'y_FAC', 'z_FAC']).assign_coords(v_dim=np.arange(3))

R_DSI_to_FAC = R_FAC_to_DSI.transpose('time', 'v_dim', 'axis')

print(R_FAC_to_DSI)
print('')
print(R_DSI_to_FAC)

In [ ]:
da_e_x_DSI_inFAC = R_DSI_to_FAC.sel(v_dim=0)
da_e_y_DSI_inFAC = R_DSI_to_FAC.sel(v_dim=1)
da_e_z_DSI_inFAC = R_DSI_to_FAC.sel(v_dim=2)

## DSI→GSM行列とFAC→GSM行列

DSIの各単位ベクトルを `erg_cotrans` でJ2000へ、`cotrans` でGSMへ変換する。
列ベクトルの規約では、変換後の3本を列に並べたものが $R_{\mathrm{GSM}\leftarrow\mathrm{DSI}}$ となる。
FACの時刻列（100 s移動平均で定義した背景磁場の有効時刻）に直接変換を評価し、行列自体の時刻補間は行わない。

既存の `R_FAC_to_DSI` / `R_DSI_to_FAC` は変更せず、FAC基底から
`R_DSI_from_FAC(time, dsi_component, fac_component)` を明示的に作る。
すべての行列は「時刻、出力成分、入力成分」の順とし、
$R_{\mathrm{GSM}\leftarrow\mathrm{FAC}}=R_{\mathrm{GSM}\leftarrow\mathrm{DSI}}R_{\mathrm{DSI}\leftarrow\mathrm{FAC}}$ と合成する。
行列は無次元。FACの軸定義（zは背景磁場、xは位置方向の磁場垂直成分、y=z×x）は既存セルを引き継ぐ。

**データ読み込み:** 以下のセルを実行するとPySPEDASがATT・軌道データを読み込む。
変換内部からダウンロードが発生する場合がある。`noload=True` は必要な内部変数の準備も省くため、ここでは指定しない。

In [ ]:
# 既存のFAC基底を、列ベクトル用の明示的な次元順序に並べる。
_fac_basis_dsi = xr.align(
    da_e_x_FAC_inDSI, da_e_y_FAC_inDSI, da_e_z_FAC_inDSI,
    join='exact',
)
R_DSI_from_FAC = xr.DataArray(
    np.stack([a.transpose('time', 'v_dim').values for a in _fac_basis_dsi], axis=-1),
    dims=('time', 'dsi_component', 'fac_component'),
    coords={
        'time': _fac_basis_dsi[0].time.values,
        'dsi_component': ['x', 'y', 'z'],
        'fac_component': ['x', 'y', 'z'],
    },
    name='R_DSI_from_FAC',
    attrs={'description': 'v_DSI = R_DSI_from_FAC @ v_FAC', 'units': '1'},
)
coordinate_matrix_atol = 1e-6  # 直交性・往復変換の絶対許容誤差
_fac_matrix = R_DSI_from_FAC.values
if _fac_matrix.shape[0] == 0 or not np.isfinite(_fac_matrix).all():
    raise ValueError('FAC基底が空、または非有限値を含む。背景磁場と位置方向を確認する。')
np.testing.assert_allclose(
    np.swapaxes(_fac_matrix, -1, -2) @ _fac_matrix,
    np.broadcast_to(np.eye(3), _fac_matrix.shape),
    atol=coordinate_matrix_atol, rtol=0,
)
np.testing.assert_allclose(np.linalg.det(_fac_matrix), 1., atol=coordinate_matrix_atol, rtol=0)
R_FAC_from_DSI = R_DSI_from_FAC.transpose('time', 'fac_component', 'dsi_component').rename('R_FAC_from_DSI')
R_FAC_from_DSI.attrs = {'description': 'v_FAC = R_FAC_from_DSI @ v_DSI', 'units': '1'}

# datetime64をUTCのUnix秒へ。PySPEDASのtplot registryを一貫して使う。
coordinate_times_unix = (
    (R_DSI_from_FAC.time.values.astype('datetime64[ns]') - np.datetime64('1970-01-01', 'ns'))
    / np.timedelta64(1, 's')
).astype(np.float64)
if not np.isfinite(coordinate_times_unix).all() or np.any(np.diff(coordinate_times_unix) <= 0):
    raise ValueError('座標変換の時刻は有限かつ単調増加である必要がある。')
print(R_DSI_from_FAC)

In [ ]:
def coordinate_dsi_to_gsm(vectors_dsi, times_unix, tag):
    """同一時刻の(N, 3)ベクトルをDSI→J2000→GSMへ変換する。"""
    vectors_dsi = np.asarray(vectors_dsi, dtype=np.float64)
    times_unix = np.asarray(times_unix, dtype=np.float64)
    if vectors_dsi.shape != (times_unix.size, 3):
        raise ValueError('vectors_dsiには時刻数と一致する(N, 3)配列を指定する。')
    names = [f'coordinate_plot_{tag}_{coord}' for coord in ('dsi', 'j2000', 'gsm')]
    # 古い出力を消し、変換失敗時に前回の結果を読むことを防ぐ。
    for name in names:
        psp.del_data(name)
    try:
        psp.store_data(names[0], data={'x': times_unix, 'y': vectors_dsi})
        psp.projects.erg.erg_cotrans(
            in_name=names[0], out_name=names[1], in_coord='dsi', out_coord='j2000',
        )
        if psp.get_data(names[1]) is None:
            raise RuntimeError('DSI→J2000変換に失敗した。ATT・軌道データを確認する。')
        psp.cotrans(name_in=names[1], name_out=names[2], coord_in='j2000', coord_out='gsm')
        result = psp.get_data(names[2])
        if result is None:
            raise RuntimeError('J2000→GSM変換に失敗した。')
        if not np.array_equal(result.times, times_unix):
            raise RuntimeError('座標変換の前後で時刻列が一致しない。')
        transformed = np.array(result.y, dtype=np.float64, copy=True)
        if transformed.shape != vectors_dsi.shape or not np.isfinite(transformed).all():
            raise ValueError('変換結果の形状または有限性が不正。ATT・軌道の時間範囲と欠損を確認する。')
        return transformed
    finally:
        # この関数が作成した一時ベクトルだけを削除する。
        for name in names:
            psp.del_data(name)


_dsi_basis_gsm = []
for i, axis_label in enumerate(('x', 'y', 'z')):
    unit_vector = np.broadcast_to(np.eye(3)[i], (coordinate_times_unix.size, 3)).copy()
    _dsi_basis_gsm.append(coordinate_dsi_to_gsm(unit_vector, coordinate_times_unix, f'dsi_basis_{axis_label}'))

R_DSI_to_GSM = xr.DataArray(
    np.stack(_dsi_basis_gsm, axis=-1),
    dims=('time', 'gsm_component', 'dsi_component'),
    coords={
        'time': R_DSI_from_FAC.time.values,
        'gsm_component': ['x', 'y', 'z'],
        'dsi_component': ['x', 'y', 'z'],
    },
    name='R_DSI_to_GSM',
    attrs={'description': 'v_GSM = R_DSI_to_GSM @ v_DSI', 'units': '1'},
)
R_GSM_to_DSI = R_DSI_to_GSM.transpose('time', 'dsi_component', 'gsm_component').rename('R_GSM_to_DSI')
R_GSM_to_DSI.attrs = {'description': 'v_DSI = R_GSM_to_DSI @ v_GSM', 'units': '1'}

# 共通のDSI成分について縮約。xarrayの次元名で入力・出力を区別する。
R_FAC_to_GSM = xr.dot(R_DSI_to_GSM, R_DSI_from_FAC, dim='dsi_component')
R_FAC_to_GSM = R_FAC_to_GSM.transpose('time', 'gsm_component', 'fac_component').rename('R_FAC_to_GSM')
R_FAC_to_GSM.attrs = {'description': 'v_GSM = R_FAC_to_GSM @ v_FAC', 'units': '1'}
R_GSM_to_FAC = R_FAC_to_GSM.transpose('time', 'fac_component', 'gsm_component').rename('R_GSM_to_FAC')
R_GSM_to_FAC.attrs = {'description': 'v_FAC = R_GSM_to_FAC @ v_GSM', 'units': '1'}
print(R_DSI_to_GSM)
print(R_FAC_to_GSM)

### 行列の検証

DSI→GSMとFAC→GSMの直交性・行列式・往復変換を確認する。
加えて、DSI表示のFAC基底3本をPySPEDASで直接GSMへ変換し、合成行列の各列と比較する。
これは行列の列配置と合成の検証であり、PySPEDASの姿勢・座標モデル自体を独立に検証するものではない。
ATT欠損区間や端点外挿の妥当性はこの数値検証だけでは保証されない。

In [ ]:
coordinate_matrix_checks = {}
for matrix in (R_DSI_to_GSM, R_FAC_to_GSM):
    values = matrix.values
    identity = np.broadcast_to(np.eye(3), values.shape)
    gram = np.swapaxes(values, -1, -2) @ values
    det = np.linalg.det(values)
    np.testing.assert_allclose(gram, identity, atol=coordinate_matrix_atol, rtol=0)
    np.testing.assert_allclose(det, 1., atol=coordinate_matrix_atol, rtol=0)
    # 各軸が非ゼロのベクトルで往復変換を確認する。
    vector_in = np.broadcast_to(np.array([0.3, -0.4, 0.5]), (values.shape[0], 3))
    vector_out = np.einsum('tij,tj->ti', values, vector_in)
    vector_back = np.einsum('tji,tj->ti', values, vector_out)
    np.testing.assert_allclose(vector_back, vector_in, atol=coordinate_matrix_atol, rtol=0)
    coordinate_matrix_checks[matrix.name] = {
        'max_orthogonality_error': float(np.max(np.abs(gram - identity))),
        'max_determinant_error': float(np.max(np.abs(det - 1.))),
        'max_roundtrip_error': float(np.max(np.abs(vector_back - vector_in))),
    }

_fac_basis_gsm_direct = np.stack([
    coordinate_dsi_to_gsm(
        R_DSI_from_FAC.sel(fac_component=axis_label).values,
        coordinate_times_unix, f'fac_basis_{axis_label}',
    )
    for axis_label in ('x', 'y', 'z')
], axis=-1)
np.testing.assert_allclose(
    R_FAC_to_GSM.values, _fac_basis_gsm_direct,
    atol=coordinate_matrix_atol, rtol=0,
)
coordinate_matrix_checks['FAC_direct_vs_composed_max_error'] = float(
    np.max(np.abs(R_FAC_to_GSM.values - _fac_basis_gsm_direct))
)
print(coordinate_matrix_checks)
print('座標変換行列の数値検証に合格')

# 使用例: FAC成分を持つDataArray v_fac(time, fac_component)に対して
# v_gsm = xr.dot(R_FAC_to_GSM, v_fac, dim='fac_component').transpose('time', 'gsm_component')
# 時刻が異なるデータへ適用する場合は、その時刻で行列を作成する。

In [ ]:
# FAC→GSM行列の各列は、GSM表示のFAC単位ベクトル。
# fac_componentでFAC軸を選び、gsm_component（x, y, z）を残す。
da_e_x_FAC_inGSM = R_FAC_to_GSM.sel(fac_component='x', drop=True).rename('da_e_x_FAC_inGSM')
da_e_y_FAC_inGSM = R_FAC_to_GSM.sel(fac_component='y', drop=True).rename('da_e_y_FAC_inGSM')
da_e_z_FAC_inGSM = R_FAC_to_GSM.sel(fac_component='z', drop=True).rename('da_e_z_FAC_inGSM')

for basis in (da_e_x_FAC_inGSM, da_e_y_FAC_inGSM, da_e_z_FAC_inGSM):
    basis.attrs = {'description': 'FAC unit vector expressed in GSM', 'units': '1', 'coordinate_system': 'GSM'}
    print(basis)


In [ ]:
# 共通定義: θ=asin(e_Z), ξ=atan2(e_Y, e_X)。単位はdegree。
# θはGSM XY平面からの仰角、ξは+Xから+Y方向を正とする方位角。
# 位置データは角度計算に不要。FAC基底の構成方法は変更しない。
angle_singularity_tol = 1e-10


def make_fac_orientation_angles(fac_z_gsm, singularity_tol=1e-10):
    """FAC-ZのGSM XY平面からの仰角θと、+X→+Yの方位角ξ（degree）。"""
    ez_da = fac_z_gsm.transpose('time', 'gsm_component').sel(gsm_component=['x', 'y', 'z'])
    times = ez_da.time.values.astype('datetime64[ns]')
    if times.size == 0 or np.isnat(times).any() or np.any(np.diff(times) <= np.timedelta64(0, 'ns')):
        raise ValueError('FAC時刻は空でなく、NaTを含まず単調増加である必要がある。')
    if not np.isfinite(singularity_tol) or not 0 < singularity_tol < 1:
        raise ValueError('singularity_tolは0より大きく1未満を指定する。')
    raw = np.asarray(ez_da.values, dtype=np.float64)
    norm = np.linalg.norm(raw, axis=1)
    valid = np.isfinite(raw).all(axis=1) & np.isfinite(norm) & (norm > 0)
    ez = np.full_like(raw, np.nan)
    ez[valid] = raw[valid] / norm[valid, None]
    theta = np.rad2deg(np.arcsin(np.clip(ez[:, 2], -1., 1.)))
    # XY投影が消える場合、仰角は±90°だが方位角は定義できない。
    xi_valid = valid & (np.hypot(ez[:, 0], ez[:, 1]) > singularity_tol)
    xi = np.full(theta.shape, np.nan)
    azimuth_deg = np.rad2deg(np.arctan2(ez[xi_valid, 1], ez[xi_valid, 0]))
    xi[xi_valid] = (azimuth_deg + 90.) % 360. - 90.
    # 丸めで上端になった場合も[-90, 270)へ収める。
    xi[xi >= 270.] = -90.
    angles = xr.Dataset(
        {'theta_deg': ('time', theta), 'xi_deg': ('time', xi)},
        coords={'time': ez_da.time.values},
        attrs={
            'coordinate_system': 'GSM',
            'definition_id': 'gsm_xy_elevation_azimuth_m90_270_v1',
            'definition': 'theta=asin(e_z.GSM_Z); xi=(atan2(e_z.GSM_Y, e_z.GSM_X) degrees + 90) mod 360 - 90 degrees',
            'singularity_tol': singularity_tol,
        },
    )
    angles.theta_deg.attrs = {
        'units': 'degree', 'valid_range': [-90., 90.],
        'description': 'Signed elevation of FAC-Z above GSM XY plane; positive toward +Z',
    }
    angles.xi_deg.attrs = {
        'units': 'degree', 'valid_range': [-90., 270.],
        'description': 'Azimuth of FAC-Z projected onto GSM XY plane in [-90, 270) degrees; positive from +X toward +Y',
    }
    return angles


fac_orientation_angles = make_fac_orientation_angles(
    da_e_z_FAC_inGSM, singularity_tol=angle_singularity_tol,
)
print(fac_orientation_angles)


In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
mpl.rcParams['font.size'] = 15

coordinate_save_dir = (
    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/2230-2330/coordinate/FAC_in_GSM"
)


def fac_basis_gsm_at(it):
    basis = xr.align(da_e_x_FAC_inGSM, da_e_y_FAC_inGSM, da_e_z_FAC_inGSM, join='exact')
    vectors = np.stack([
        da.isel(time=it).sel(gsm_component=['x', 'y', 'z']).values for da in basis
    ])  # (FAC軸, GSM成分)。配列内の座標ラベルは既存の小文字を維持。
    if not np.isfinite(vectors).all():
        raise ValueError(f'FAC基底に非有限値がある: time index={it}')
    return vectors


def setup_polar_ax(ax, horizontal, vertical):
    ax.set_title(f'GSM ({horizontal}, {vertical}) plane')
    ax.set_theta_zero_location('E')
    ax.set_theta_direction(1)
    ax.set_thetagrids(np.arange(0, 360, 45))
    minor_deg = np.arange(0, 360, 15)
    ax.set_xticks(np.deg2rad(minor_deg[minor_deg % 45 != 0]), minor=True)
    ax.tick_params(axis='x', which='minor', labelbottom=False)
    ax.set_ylim(0, 1.05)
    ax.set_rticks([0.25, 0.5, 0.75, 1.0])
    ax.set_rlabel_position(135)
    ax.grid(True, which='major')
    ax.xaxis.grid(True, which='minor', linestyle=':', linewidth=0.6, alpha=0.6)


def cart_to_polar(u, v):
    return np.arctan2(v, u), np.hypot(u, v)


def plot_polar_vector(ax, u, v, color, label):
    theta, r = cart_to_polar(u, v)
    ax.plot([theta, theta], [0, r], color=color, lw=2, label=label)
    ax.plot(theta, r, 'o', color=color, ms=5)


def setup_ax(ax, horizontal, vertical):
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_xlabel(f'GSM-{horizontal}')
    ax.set_ylabel(f'GSM-{vertical}')
    ax.minorticks_on()
    ax.grid(True, which='both', alpha=0.3)
    ax.set_title(f'GSM ({horizontal}, {vertical}) plane')


def angle_line_with_breaks(times, values, wrapped=False):
    """ξの−90°/270°境界と時間欠損にNaNを挿入し、両側の有効点を保持する。"""
    times = np.asarray(times).astype('datetime64[ns]')
    values = np.asarray(values, dtype=float)
    if times.size < 2:
        return times, values
    dt = np.diff(times) / np.timedelta64(1, 's')
    breaks = dt > 3 * np.median(dt)
    if wrapped:
        breaks |= np.abs(np.diff(values)) > 180.
    indices = np.flatnonzero(breaks) + 1
    return np.insert(times, indices, times[indices]), np.insert(values, indices, np.nan)


def add_orientation_time_series(fig, gridspec, snapshot_time, time_range=None, angle_limits=None):
    angles = fac_orientation_angles
    if angles.attrs.get('definition_id') != 'gsm_xy_elevation_azimuth_m90_270_v1':
        raise ValueError('角度の定義が旧版。GSM XY基準の角度計算セルを再実行する。')
    # 古い角度計算結果を誤って別のFAC時系列に対応させない。
    if not np.array_equal(angles.time.values, da_e_z_FAC_inGSM.time.values):
        raise ValueError('角度とFACの時刻列が異なる。角度計算セルを再実行する。')
    if time_range is not None:
        angles = angles.sel(time=slice(*time_range))
        if angles.sizes['time'] == 0:
            raise ValueError('指定時間範囲に角度データがない。')
    ax_theta = fig.add_subplot(gridspec[1, :])
    ax_xi = fig.add_subplot(gridspec[2, :], sharex=ax_theta)
    for ax, key, color, panel, ylabel, ylim in (
        (ax_theta, 'theta_deg', 'tab:blue', '(i)', r'$\theta$ [deg]', (-90, 90)),
        (ax_xi, 'xi_deg', 'tab:orange', '(j)', r'$\xi$ [deg]', (-90, 270)),
    ):
        t, values = angle_line_with_breaks(angles.time.values, angles[key].values, wrapped=(key == 'xi_deg'))
        ax.plot(t, values, color=color, lw=1.2)
        ax.axvline(snapshot_time, color='red', linestyle='--', lw=1.5)
        ax.set_ylabel(ylabel)
        ax.set_ylim(*(angle_limits[key] if angle_limits is not None else ylim))
        ax.minorticks_on()
        ax.grid(True, which='both', alpha=0.3)
        ax.text(0.01, 0.96, panel, transform=ax.transAxes, ha='left', va='top', fontweight='bold')
    if angle_limits is None:
        ax_theta.set_yticks([-90, -45, 0, 45, 90])
        ax_xi.set_yticks([-90, 0, 90, 180, 270])
    else:
        for ax in (ax_theta, ax_xi):
            ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=5))
    if time_range is not None:
        ax_xi.set_xlim(*time_range)
    elif angles.sizes['time'] > 1:
        ax_xi.set_xlim(angles.time.values[0], angles.time.values[-1])
    locator = mdates.AutoDateLocator(minticks=4, maxticks=10)
    ax_xi.xaxis.set_major_locator(locator)
    ax_xi.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    ax_theta.tick_params(axis='x', labelbottom=False)
    ax_xi.set_xlabel('Time [UTC]')
    return ax_theta, ax_xi


def make_fac_gsm_figure(it, style='polar', time_range=None, angle_limits=None):
    """上段(f)-(h): FAC基底、中段(i): θ、下段(j): ξ。"""
    if style not in ('polar', 'cartesian'):
        raise ValueError("styleは'polar'または'cartesian'を指定する。")
    vectors = fac_basis_gsm_at(it)
    fig = plt.figure(figsize=(16, 11), layout='constrained')
    fig.get_layout_engine().set(h_pad=0.12, w_pad=0.08)
    gs = fig.add_gridspec(3, 3, height_ratios=[2.8, 1, 1], hspace=0.10)
    snapshot_time = da_e_x_FAC_inGSM.time.values[it]
    for col, ((i, j), panel) in enumerate(zip(((0, 1), (0, 2), (1, 2)), ('(f)', '(g)', '(h)'))):
        ax = fig.add_subplot(gs[0, col], projection='polar' if style == 'polar' else None)
        horizontal, vertical = 'XYZ'[i], 'XYZ'[j]
        if style == 'polar':
            setup_polar_ax(ax, horizontal, vertical)
            plot_polar_vector(ax, 1, 0, 'k', f'GSM-{horizontal}')
            plot_polar_vector(ax, 0, 1, 'gray', f'GSM-{vertical}')
            for vector, color, label in zip(vectors, ('r', 'b', 'g'), ('FAC-x', 'FAC-y', 'FAC-z')):
                plot_polar_vector(ax, vector[i], vector[j], color, label)
        else:
            setup_ax(ax, horizontal, vertical)
            for u, v, color, label in ((1, 0, 'k', f'GSM-{horizontal}'), (0, 1, 'gray', f'GSM-{vertical}')):
                ax.quiver(0, 0, u, v, angles='xy', scale_units='xy', scale=1, color=color, label=label)
            for vector, color, label in zip(vectors, ('r', 'b', 'g'), ('FAC-x', 'FAC-y', 'FAC-z')):
                ax.quiver(0, 0, vector[i], vector[j], angles='xy', scale_units='xy', scale=1, color=color, label=label)
        ax.text(-0.08, 1.10, panel, transform=ax.transAxes, ha='left', va='top', fontweight='bold')
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=3, fontsize=15)
    add_orientation_time_series(fig, gs, snapshot_time, time_range=time_range, angle_limits=angle_limits)
    time_label = np.datetime_as_string(snapshot_time, unit='ms')
    fig.suptitle(f'Arase | FAC unit vectors in GSM | {time_label} UTC', fontsize=20)
    return fig


def save_fac_gsm_frame(it, save_dir, style, time_range=None, angle_limits=None):
    fig = make_fac_gsm_figure(it, style=style, time_range=time_range, angle_limits=angle_limits)
    time_label = np.datetime_as_string(da_e_x_FAC_inGSM.time.values[it], unit='ms')
    time_stamp = time_label.replace('-', '').replace(':', '').replace('T', '_').replace('.', '_')
    os.makedirs(save_dir, exist_ok=True)
    fname = os.path.join(save_dir, f'fac_in_gsm_{style}_{time_stamp}UTC.png')
    try:
        fig.savefig(fname, dpi=150)
    finally:
        plt.close(fig)
    return fname


def plot_fac_gsm_frame_polar(it, frame_idx, save_dir):
    return save_fac_gsm_frame(it, save_dir, style='polar')


def select_minute_frames(times, tolerance_sec=None, target_start=None, target_end=None):
    """毎分00秒に最も近いサンプルを選ぶ。戻り値のoffsetは実時刻−00秒（秒）。"""
    times = np.asarray(times).astype('datetime64[ns]')
    if times.ndim != 1 or times.size == 0 or np.isnat(times).any():
        raise ValueError('時刻は空でない1次元配列で、NaTを含まない必要がある。')
    dt_ns = np.diff(times.astype(np.int64))
    if np.any(dt_ns <= 0):
        raise ValueError('時刻は単調増加である必要がある。')
    # 既定許容差はサンプル間隔の中央値。長い欠損区間をまたいで選ばない。
    if tolerance_sec is None:
        tolerance_sec = float(np.median(dt_ns) / 1e9) if dt_ns.size else 0.
    if not np.isfinite(tolerance_sec) or tolerance_sec < 0:
        raise ValueError('tolerance_secは非負の有限値を指定する。')
    first_minute = times[0].astype('datetime64[m]').astype('datetime64[ns]')
    if first_minute < times[0]:
        first_minute += np.timedelta64(1, 'm')
    if target_start is not None:
        first_minute = np.datetime64(target_start, 'ns')
    last_time = times[-1] if target_end is None else np.datetime64(target_end, 'ns')
    if np.isnat(first_minute) or np.isnat(last_time) or ((target_start is not None or target_end is not None) and last_time < first_minute):
        raise ValueError('毎分00秒の選択対象範囲が不正。')
    targets = np.arange(first_minute, last_time + np.timedelta64(1, 'ns'),
                        np.timedelta64(1, 'm'))
    right = np.clip(np.searchsorted(times, targets), 0, times.size - 1)
    left = np.maximum(right - 1, 0)
    # 同距離なら前のサンプルを選ぶ。
    indices = np.where(np.abs(times[left] - targets) <= np.abs(times[right] - targets), left, right)
    offsets = (times[indices] - targets) / np.timedelta64(1, 's')
    keep = np.abs(offsets) <= tolerance_sec
    return targets, indices, offsets, keep, tolerance_sec




In [ ]:
# 直交座標版も同じ3段構成と角度時系列を使う。
def plot_fac_gsm_frame(it, frame_idx, save_dir):
    return save_fac_gsm_frame(it, save_dir, style='cartesian')

# 1枚だけ画面で確認する場合（全フレームの保存は次のセル）：
# fig = make_fac_gsm_figure(it=0, style='polar')
# plt.show()


In [ ]:
from tqdm.auto import tqdm


plot_style = 'polar'  # 'polar' または 'cartesian'
minute_tolerance_sec = None  # None: サンプル間隔の中央値。必要なら秒単位で明示指定。
plotters = {'polar': plot_fac_gsm_frame_polar, 'cartesian': plot_fac_gsm_frame}
if plot_style not in plotters:
    raise ValueError("plot_styleは'polar'または'cartesian'を指定する。")

frame_times = da_e_x_FAC_inGSM.time.values.astype('datetime64[ns]')
target_times, candidate_indices, time_offsets_sec, keep_frames, used_tolerance_sec = select_minute_frames(
    frame_times, tolerance_sec=minute_tolerance_sec,
)
frame_indices = candidate_indices[keep_frames]
# 毎分00秒の目標と実際に使用するサンプルの対応を保持する。
frame_selection = xr.Dataset({
    'target_time': ('frame', target_times[keep_frames]),
    'sample_time': ('frame', frame_times[frame_indices]),
    'time_offset_sec': ('frame', time_offsets_sec[keep_frames]),
    'sample_index': ('frame', frame_indices),
})
frame_selection['time_offset_sec'].attrs = {'units': 's', 'description': 'sample_time - target_time'}
print('num frames:', frame_indices.size)
print('skipped minute targets:', int((~keep_frames).sum()))
print('nearest-sample tolerance [s]:', used_tolerance_sec)
print('save directory:', coordinate_save_dir)
print(frame_selection)

# 図のタイトルは使用した実サンプルのUTC時刻。基底の補間・再正規化は行わない。
plot_frame = plotters[plot_style]
for frame_idx, it in enumerate(tqdm(frame_indices, desc=f'FAC in GSM ({plot_style}, minute 00)')):
    plot_frame(int(it), frame_idx, coordinate_save_dir)


In [ ]:
# 2022-09-01 22:25–23:15 UTC専用。両端の毎分00秒を含む（最大51枚）。
# 前提: 角度計算・描画関数の定義セルまで実行済み。全区間保存セルの実行は不要。
from tqdm.auto import tqdm

range_start = np.datetime64('2022-09-01T22:25:00', 'ns')
range_end = np.datetime64('2022-09-01T23:15:00', 'ns')
range_plot_style = 'polar'  # 'cartesian'も指定可能
range_minute_tolerance_sec = None
range_save_dir = (
    '/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_third/2230-2330/coordinate/'
    'FAC_in_GSM_20220901_2225-2315_UTC'
)


def angle_limits_for_window(angles, time_start, time_end, padding_fraction=0.05, min_padding_deg=0.5):
    """区間全体の有限値から余白付き範囲を決める。全フレームで共通に使う。"""
    if padding_fraction < 0 or min_padding_deg <= 0:
        raise ValueError('余白率は非負、最小余白は正にする。')
    selected = angles.sel(time=slice(time_start, time_end))
    limits = {}
    for key, physical_range in (('theta_deg', (-90., 90.)), ('xi_deg', (-90., 270.))):
        values = selected[key].values
        values = values[np.isfinite(values)]
        if values.size == 0:
            raise ValueError(f'{key}: 指定区間に有効値がない。FACデータを確認する。')
        low, high = float(values.min()), float(values.max())
        pad = max((high - low) * padding_fraction, min_padding_deg)
        limits[key] = (max(physical_range[0], low - pad), min(physical_range[1], high + pad))
    return limits


if range_plot_style not in ('polar', 'cartesian'):
    raise ValueError("range_plot_styleは'polar'または'cartesian'を指定する。")
if range_end <= range_start:
    raise ValueError('時間範囲の終了は開始より後にする。')
range_angle_limits = angle_limits_for_window(fac_orientation_angles, range_start, range_end)
range_all_times = da_e_x_FAC_inGSM.time.values.astype('datetime64[ns]')
# 実サンプルもこの区間内から選ぶ。境界を越えた点は保存しない。
range_global_indices = np.flatnonzero((range_all_times >= range_start) & (range_all_times <= range_end))
if range_global_indices.size == 0:
    raise ValueError('指定区間にFACサンプルがない。')
range_targets, range_candidates, range_offsets, range_keep, range_used_tolerance_sec = select_minute_frames(
    range_all_times[range_global_indices], tolerance_sec=range_minute_tolerance_sec,
    target_start=range_start, target_end=range_end,
)
range_frame_indices = range_global_indices[range_candidates[range_keep]]
range_frame_selection = xr.Dataset({
    'target_time': ('frame', range_targets[range_keep]),
    'sample_time': ('frame', range_all_times[range_frame_indices]),
    'time_offset_sec': ('frame', range_offsets[range_keep]),
    'sample_index': ('frame', range_frame_indices),
})
range_frame_selection.time_offset_sec.attrs['units'] = 's'
print('time range [UTC]:', range_start, 'to', range_end)
print('angle limits [deg]:', range_angle_limits)
print('num frames:', range_frame_indices.size, '/ minute targets:', range_targets.size)
print('nearest-sample tolerance [s]:', range_used_tolerance_sec)
print('save directory:', range_save_dir)
if range_frame_indices.size == 0:
    raise ValueError('毎分00秒の許容差内にサンプルがない。保存は行わない。')
os.makedirs(range_save_dir, exist_ok=True)
for it in tqdm(range_frame_indices, desc='Arase 22:25–23:15 UTC'):
    save_fac_gsm_frame(
        int(it), range_save_dir, range_plot_style,
        time_range=(range_start, range_end), angle_limits=range_angle_limits,
    )
